In [1]:
from freqtrade.configuration import Configuration
from pathlib import Path
import os
from freqtrade.data.history import load_pair_history
from freqtrade.enums import CandleType
from freqtrade.resolvers import StrategyResolver
from freqtrade.data.dataprovider import DataProvider
from plotly import graph_objects as go
from freqtrade.plot.plotting import  generate_candlestick_graph
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
from pandas import DataFrame
from freqtrade.strategy import IStrategy, merge_informative_pair

In [2]:
project_root = "somedir/freqtrade"
i=0
try:
    os.chdirdir(project_root)
    assert Path('docker-compose.yml').is_file()
except:
    while i<4 and (not Path('docker-compose.yml').is_file()):
        os.chdir(Path(Path.cwd(), '../'))
        i+=1
    project_root = Path.cwd()
print(Path.cwd())

/root/defi


In [10]:
config = Configuration.from_files(["user_data/cluster_strategy_v1_config.json"])
config["strategy"] = "ClusterStrategyV1"
data_location = config["datadir"]
pair = 'WIF/USDT:USDT'

In [ ]:
!freqtrade download-data -c user_data/cluster_strategy_v1_config.json --pairs $pair -t 1m

In [13]:
dataframe_1m = load_pair_history(
    datadir=data_location,
    timeframe='1m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)

In [14]:
dataframe_5m = load_pair_history(
    datadir=data_location,
    timeframe='5m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)

In [50]:
def cluster_borders(pair):
    dataframe = dataframe_5m.iloc[dataframe_5m.index > 10]
    X = dataframe.close.values.reshape(-1,1)
    kmeans = KMeans(n_clusters=6, random_state=42).fit(X)
    dataframe['cluster'] = kmeans.predict(X)
    return dataframe.groupby(['cluster']).min().close.sort_values().values

In [57]:
dataframe_5m.iloc[dataframe_5m.index > dataframe_5m.index.iat[-2]]

AttributeError: 'RangeIndex' object has no attribute 'iat'

In [ ]:
borders = cluster_borders(pair)

for c, border in enumerate(borders):
    dataframe_1m.loc[(dataframe_1m.close >= border),'cluster'] = c

dataframe_1m

In [38]:
dataframe_1m[(dataframe_1m.cluster.shift(1) == 0) & (dataframe_1m.cluster == 1)]

,date,open,high,low,close,volume,cluster
404,2024-03-26 06:44:00+00:00,2.9324,2.9388,2.9311,2.9344,36911.0,1.0
416,2024-03-26 06:56:00+00:00,2.9306,2.9530,2.9258,2.9409,94452.0,1.0
431,2024-03-26 07:11:00+00:00,2.9300,2.9508,2.9297,2.9401,35900.0,1.0
461,2024-03-26 07:41:00+00:00,2.9253,2.9436,2.9206,2.9396,35568.0,1.0
477,2024-03-26 07:57:00+00:00,2.9318,2.9403,2.9255,2.9373,185608.0,1.0
...,...,...,...,...,...,...,...
38980,2024-04-22 01:40:00+00:00,2.9259,2.9375,2.9166,2.9354,50621.0,1.0
39067,2024-04-22 03:07:00+00:00,2.9220,2.9372,2.9219,2.9357,77946.0,1.0
39084,2024-04-22 03:24:00+00:00,2.9331,2.9383,2.9320,2.9383,8932.0,1.0
39602,2024-04-22 12:02:00+00:00,2.9322,2.9356,2.9251,2.9348,40377.0,1.0


In [41]:
dataframe_1m.loc[
    (
        (dataframe_1m.cluster.shift(1) == 0) &
        (dataframe_1m.cluster == 1)
    ),
    'enter_long'
] = 1

dataframe_1m.loc[
    (
        (dataframe_1m.cluster.shift(1) == 5) &
        (dataframe_1m.cluster == 4)
    ),
    'enter_short'
] = 1

In [44]:
dataframe_1m[dataframe_1m.enter_short==1]

,date,open,high,low,close,volume,cluster,new,enter_long,enter_short
503,2024-03-26 08:23:00+00:00,2.9750,2.9762,2.9633,2.9648,38244.0,4.0,2.9750,NaN,1.0
512,2024-03-26 08:32:00+00:00,2.9756,2.9763,2.9608,2.9673,63204.0,4.0,2.9756,NaN,1.0
3092,2024-03-28 03:32:00+00:00,2.9767,2.9852,2.9617,2.9662,88322.0,4.0,2.9767,NaN,1.0
3111,2024-03-28 03:51:00+00:00,2.9779,2.9790,2.9641,2.9674,37722.0,4.0,2.9779,NaN,1.0
28392,2024-04-14 17:12:00+00:00,3.0135,3.0162,2.9603,2.9624,311864.0,4.0,3.0135,NaN,1.0
28691,2024-04-14 22:11:00+00:00,2.9786,2.9820,2.9564,2.9615,135962.0,4.0,2.9786,NaN,1.0
28836,2024-04-15 00:36:00+00:00,2.9806,2.9811,2.9621,2.9631,73573.0,4.0,2.9806,NaN,1.0
28838,2024-04-15 00:38:00+00:00,2.9764,2.9765,2.9622,2.9647,60793.0,4.0,2.9764,NaN,1.0
28842,2024-04-15 00:42:00+00:00,2.9740,2.9781,2.9652,2.9682,42003.0,4.0,2.9740,NaN,1.0
29494,2024-04-15 11:34:00+00:00,2.9744,2.9761,2.9582,2.9668,101608.0,4.0,2.9744,NaN,1.0


In [28]:
dataframe = load_pair_history(
    datadir=data_location,
    timeframe='4h',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
dataframe = dataframe[-6:]
dataframe['pct_change'] = abs(dataframe.close.pct_change())
mean = dataframe['pct_change'].mean()
dataframe_ = load_pair_history(
    datadir=data_location,
    timeframe='5m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
dataframe_ = dataframe_[-288:]
dataframe_['pct_change_'] = abs(dataframe_.close.pct_change())
mean_ = dataframe_['pct_change_'].mean()
n = int(mean / mean_)

8

In [15]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=dataframe['date'],y=dataframe['4h_close_pct_change'], mode = 'markers'))
fig.show()

In [19]:
df = dataframe[dataframe.date>='2024-04-02']

In [10]:
def cluster_borders(dataframe, n):
    X = dataframe.close.values.reshape(-1,1)
    kmeans = KMeans(n_clusters=n, random_state=42).fit(X)
    dataframe['cluster'] = kmeans.predict(X)
    borders = dataframe.groupby(['cluster']).min().close.sort_values().values
    return np.append(borders, dataframe.close.max())

In [29]:
dataframe = load_pair_history(
    datadir=data_location,
    timeframe='1m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
dataframe = dataframe[-240:]

In [30]:
borders = cluster_borders(dataframe, 8)
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for border in borders:
    fig.add_hline(y=border, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [61]:
borders

array([2.6589, 2.6888, 2.7067, 2.7287, 2.7593, 2.7908, 2.8241])

In [73]:
1 - borders[0]/borders[3]

0.05849651216316698

In [50]:
borders = cluster_borders()
fig = generate_candlestick_graph(pair=pair, data=df)
for border in borders:
    fig.add_hline(y=border, line_width=1, line_color='green')
# fig.add_hline(y=2.577, line_width=1, line_color='red')
# fig.add_hline(y=3.5767, line_width=1, line_color='orange')
# fig.add_vline(x="2024-04-11 12:33:35", line_width=1, line_color="orange")
fig.add_hline(y=2.5853, line_width=1, line_color='blue')
fig.add_vline(x="2024-04-14 05:10:02", line_width=1, line_color="blue")
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

/tmp/ipykernel_1405489/3696386203.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [14]:
from sklearn.cluster import HDBSCAN

In [46]:
df_ = dataframe.copy()
X = df_['close'].values.reshape(-1,1)
hdb = HDBSCAN(min_cluster_size=20)
hdb.fit(X)
df_['label'] = hdb.labels_
df_['probability'] = hdb.probabilities_

In [48]:
mins = df_[(df_.label!=-1) & (df_.probability==1)].groupby(['label']).min().close.values
maxs = df_[(df_.label!=-1) & (df_.probability==1)].groupby(['label']).max().close.values
fig = generate_candlestick_graph(pair=pair, data=df)
for border in mins:
    fig.add_hline(y=border, line_width=1, line_color='red')
for border in maxs:
    fig.add_hline(y=border, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [40]:
df_[(df_.label!=-1) & (df_.probability==1)].groupby(['label']).min()

,date,open,high,low,close,volume,probability
label,,,,,,,
0,2024-03-31 10:00:00+00:00,4.7035,4.7342,4.6500,4.7152,2077615.0,1.0
1,2024-03-30 12:00:00+00:00,4.3401,4.4872,4.3188,4.4809,1752583.0,1.0
2,2024-03-30 04:00:00+00:00,4.2844,4.5900,4.2840,4.5380,2532632.0,1.0
3,2024-03-20 04:00:00+00:00,2.1144,2.1388,1.8156,2.1140,4191889.0,1.0
4,2024-03-30 15:00:00+00:00,4.2413,4.3583,4.2058,4.3366,2438636.0,1.0
5,2024-03-15 18:00:00+00:00,2.8373,3.0600,2.8363,3.0427,2821952.0,1.0
6,2024-03-18 13:00:00+00:00,3.0901,3.1259,3.0158,3.0883,1340283.0,1.0
7,2024-03-27 08:00:00+00:00,3.1055,3.1739,3.0886,3.1371,2385916.0,1.0
8,2024-03-28 18:00:00+00:00,3.3496,3.4283,3.3306,3.4173,1137663.0,1.0


In [76]:
dataframe = load_pair_history(
    datadir=data_location,
    timeframe='1m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
informative = load_pair_history(
    datadir=data_location,
    timeframe='1d',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
informative_2 = load_pair_history(
    datadir=data_location,
    timeframe='15m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
dataframe = merge_informative_pair(dataframe, informative, '1m', '1d', ffill=False)

In [17]:
borders.tofile('user_data/notebooks/levels.csv', sep = ',')
np.genfromtxt('user_data/notebooks/levels.csv', delimiter=',')

In [ ]:
labels = [1,2,3]
colors = ['green', 'red', 'orange']
fig = generate_candlestick_graph(pair=pair, data=dataframe)

# max_values = dataframe.groupby(['label_1']).max().close.values
# min_values = dataframe.groupby(['label_1']).min().close.values
# for max_value in max_values:
#     fig.add_hline(y=max_value, line_width=2, line_color='green')
# for min_value in min_values:
#     fig.add_hline(y=min_value, line_width=2, line_color='green')

# max_values = dataframe.groupby(['label_1', 'label_2']).max().close.values
# min_values = dataframe.groupby(['label_1', 'label_2']).min().close.values
# for max_value in max_values:
#     fig.add_hline(y=max_value, line_width=1, line_color='red')
# for min_value in min_values:
#     fig.add_hline(y=min_value, line_width=1, line_color='red')

max_values = dataframe.groupby(['label_1', 'label_2', 'label_3']).max().close.values
min_values = dataframe.groupby(['label_1', 'label_2', 'label_3']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=0.5, line_color='orange')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=0.5, line_color='orange')

fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
c_min = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.min()
c_max = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.max()
timeframe = '4h'
dataframe = prepare_dataframe(timeframe)
dataframe = dataframe[(dataframe.close >= c_min) & (dataframe.close <= c_max)]
levels = pair_levels(dataframe)
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
c_min = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.min()
c_max = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.max()
timeframe = '15m'
dataframe = prepare_dataframe(timeframe)
dataframe = dataframe[(dataframe.close >= c_min) & (dataframe.close <= c_max)]
levels = pair_levels(dataframe)
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [82]:
dataframe = prepare_dataframe(timeframe='1d')
X = dataframe.high.values
X = np.append(X, dataframe.low.values).reshape(-1, 1)
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
df = pd.DataFrame({'cluster':kmeans.predict(X),'values':X.flatten()})
levels = df.groupby(['cluster']).min()['values'].sort_values().values
levels = np.append(levels, df['values'].max())
dt = dataframe[dataframe.close >= levels[-2]].date.iat[0].date().strftime('%Y%m%d-')


In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
!freqtrade download-data -c user_data/bybit_config.json --timerange $dt --pairs $pair -t 1h

In [86]:
dataframe = prepare_dataframe(timeframe='1h')
X = dataframe.high.values
X = np.append(X, dataframe.low.values).reshape(-1, 1)
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
df = pd.DataFrame({'cluster':kmeans.predict(X),'values':X.flatten()})
levels = df.groupby(['cluster']).min()['values'].sort_values().values
levels = np.append(levels, df['values'].max())
dt = dataframe[dataframe.close >= levels[-3]].date.iat[0].date().strftime('%Y%m%d-')

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
!freqtrade download-data -c user_data/bybit_config.json --timerange $20240111- --pairs $pair -t 3m

In [95]:
dataframe = prepare_dataframe(timeframe='3m')

In [96]:
def cluster(dataframe, n=3):
    X = dataframe.high.values
    X = np.append(X, dataframe.low.values).reshape(-1, 1)
    kmeans = KMeans(n_clusters=n, random_state=42).fit(X)
    return kmeans.predict(X)

In [ ]:

dataframe['label_1'] = cluster(dataframe)

grouped = dataframe.groupby(['label_1']).apply(cluster).to_dict()
for c, values in grouped.items():
    condition_1 = dataframe['label_1'] == c
    dataframe.loc[condition_1, 'label_2'] = values

grouped = dataframe.groupby(['label_1','label_2']).apply(cluster).to_dict()
for c, values in grouped.items():
    condition_1 = dataframe['label_1'] == c[0]
    condition_2 = dataframe['label_2'] == c[1]
    dataframe.loc[(condition_1 & condition_2), 'label_3'] = values

In [92]:
dataframe = prepare_dataframe(timeframe='3m')
dataframe = dataframe[(dataframe.close >= levels[-3]) & (dataframe.close <= levels[-2])]
X = dataframe.high.values
X = np.append(X, dataframe.low.values).reshape(-1, 1)
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
df = pd.DataFrame({'cluster':kmeans.predict(X),'values':X.flatten()})
levels = df.groupby(['cluster']).min()['values'].sort_values().values
levels = np.append(levels, df['values'].max())
dt = dataframe[dataframe.close >= levels[-3]].date.iat[0].date().strftime('%Y%m%d-')

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [66]:
dataframe = prepare_dataframe(timeframe='1d')
# levels = pair_levels(pair)
for c, level in enumerate(levels):
    dataframe.loc[(dataframe.close >= level),'cluster'] = c

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [46]:
timerange = '20240405-'

In [ ]:
!freqtrade download-data -c user_data/bybit_config.json --timerange $timerange --pairs $pair -t 1m

In [51]:
dataframe = load_pair_history(
        datadir=data_location,
        timeframe='1m',
        pair=pair,
        data_format = "feather",
        candle_type=CandleType.FUTURES,
    )
dataframe = dataframe[dataframe.date>='2024-04-05 02:00']

In [52]:
def pair_levels_1m(pair):
    dataframe = load_pair_history(
        datadir = config["datadir"],
        timeframe = '1m',
        pair = pair,
        data_format = "feather",
        candle_type=CandleType.FUTURES,
    )
    dataframe = dataframe[dataframe.date>='2024-04-05 02:00']
    dataframe['cluster'] = cluster(dataframe)
    levels = dataframe.groupby(['cluster']).min().close.sort_values().values
    levels = np.append(levels, dataframe.close.max())
    return levels

In [53]:
levels = pair_levels_1m(pair)
for c, level in enumerate(levels):
    dataframe.loc[(dataframe.close >= level),'cluster'] = c

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
labels = [1,2,3]
colors = ['green', 'red', 'orange']
fig = generate_candlestick_graph(pair=pair, data=dataframe)

max_values = dataframe[[f'label_1', 'close']].groupby(['label_1']).max().close.values
min_values = dataframe[[f'label_1', 'close']].groupby(['label_1']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=2, line_color='green')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=2, line_color='green')

max_values = dataframe[[f'label_1', 'label_2', 'close']].groupby(['label_1', 'label_2']).max().close.values
min_values = dataframe[[f'label_1', 'label_2', 'close']].groupby(['label_1', 'label_2']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=1, line_color='red')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=1, line_color='red')

max_values = dataframe[[f'label_1', 'label_2', 'label_3', 'close']].groupby(['label_1', 'label_2', 'label_3']).max().close.values
min_values = dataframe[[f'label_1', 'label_2', 'label_3', 'close']].groupby(['label_1', 'label_2', 'label_3']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=0.5, line_color='orange')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=0.5, line_color='orange')

fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [10]:
from freqtrade.data.btanalysis import load_trades_from_db
trades = load_trades_from_db("sqlite:///user_data/cluster_strategy_dry.sqlite")

In [20]:
trades.head(1)

,pair,stake_amount,max_stake_amount,amount,open_date,close_date,open_rate,close_rate,fee_open,fee_close,...,stop_loss_ratio,min_rate,max_rate,is_open,enter_tag,leverage,is_short,open_timestamp,close_timestamp,orders
0,GMT/USDT:USDT,49.1905,49.1905,131.0,2024-03-29 05:30:04+00:00,2024-03-29 09:04:01+00:00,0.3755,0.3803,0.0001,0.0001,...,-0.036879,0.3667,0.3803,False,,1.0,True,1711690204667,1.711703e+12,"[{'amount': 131.0, 'safe_price': 0.3755, 'ft_o..."


In [ ]:
def populate_indicators(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        dataframe['label_1'] = self.cluster(dataframe)

        grouped = dataframe.groupby(['label_1']).apply(self.cluster).to_dict()
        for c, values in grouped.items():
            condition_1 = dataframe['label_1'] == c
            dataframe.loc[condition_1, 'label_2'] = values

        grouped = dataframe.groupby(['label_1','label_2']).apply(self.cluster).to_dict()
        for c, values in grouped.items():
            condition_1 = dataframe['label_1'] == c[0]
            condition_2 = dataframe['label_2'] == c[1]
            dataframe.loc[(condition_1 & condition_2), 'label_3'] = values

        return dataframe

def populate_entry_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

    dataframe.loc[
        (
            (dataframe.label_3.shift(1) == 0) &
            (dataframe.label_3 == 1)
        ),
        'enter_long'
    ] = 1

    dataframe.loc[
        (
            (dataframe.label_3.shift(1) == 2) &
            (dataframe.label_3 == 1)
        ),
        'enter_short'
    ] = 1

    return dataframe

def populate_exit_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

    return dataframe

def position_size(self, total_asset, risk):

    size = total_asset * self.total_risk / risk if risk > self.total_risk else total_asset

    return size

def custom_stake_amount(self, pair: str, current_time: datetime, current_rate: float,
                        proposed_stake: float, min_stake: Optional[float], max_stake: float,
                        leverage: float, entry_tag: Optional[str], side: str,
                        **kwargs) -> float:

    dataframe, _ = self.dp.get_analyzed_dataframe(pair=pair, timeframe=self.timeframe)
    mins = dataframe.groupby([f'label_{i}' for i in [1,2,3]]).min().close.sort_values().values
    prev_close = dataframe.close.iat[-2]
    
    if side == 'long':
        risk = 1 - mins[mins < prev_close][-2] / prev_close
    else:
        risk = mins[mins > prev_close][1] / prev_close - 1
    
    stake = self.position_size(max_stake, risk)

    return stake

def custom_stoploss(self, pair: str, trade: 'Trade', current_time: datetime,
                    current_rate: float, current_profit: float, after_fill: bool, 
                    **kwargs) -> Optional[float]:

    dataframe, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
    mins = dataframe.groupby([f'label_{i}' for i in [1,2,3]]).min().close.sort_values().values
    prev_close = dataframe.close.iat[-2]

    if trade.is_short:
        return stoploss_from_absolute(
            mins[mins > prev_close][1],
            prev_close,
            is_short=trade.is_short,
            leverage=trade.leverage
        )
    
    return stoploss_from_absolute(
            mins[mins < prev_close][-2],
            prev_close,
            is_short=trade.is_short,
            leverage=trade.leverage
        )

In [ ]:
class ClusterStrategy(IStrategy):

    INTERFACE_VERSION = 3

    can_short: bool = True

    stoploss = -0.1

    timeframe = '1m'

    total_risk = 0.01

    process_only_new_candles = True

    use_exit_signal = False

    exit_profit_only = False

    ignore_roi_if_entry_signal = False

    use_custom_stoploss = True

    position_adjustment_enable = False

    startup_candle_count: int = 300

    order_types = {
        'entry': 'limit',
        'exit': 'limit',
        'stoploss': 'limit',
        'stoploss_on_exchange': True
    }

    order_time_in_force = {
        'entry': 'GTC',
        'exit': 'GTC'
    }

    @property
    def protections(self):
        return [
            {
                "method": "StoplossGuard",
                "lookback_period_candles": 24,
                "trade_limit": 2,
                "stop_duration_candles": 4,
                "required_profit": 0.0,
                "only_per_pair": True,
                "only_per_side": False
            }
        ]

    def cluster(self, dataframe):
        X = dataframe['close'].values.reshape(-1,1)
        kmeans = KMeans(n_clusters=5, random_state=42).fit(X)
        return kmeans.predict(X)

    def pair_levels(self, pair):
        dataframe = load_pair_history(
            datadir = self.config["datadir"],
            timeframe = self.timeframe,
            pair = pair,
            data_format = "feather",
            candle_type=CandleType.FUTURES,
        )
        # dataframe = dataframe[dataframe.date>='2024-04-05 02:00']
        dataframe['cluster'] = self.cluster(dataframe)
        levels = dataframe.groupby(['cluster']).min().close.sort_values().values
        levels = np.append(levels, dataframe.close.max())
        return levels

    def populate_indicators(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        levels = self.pair_levels(metadata['pair'])
        for c, level in enumerate(levels):
            dataframe.loc[(dataframe.close >= level),'cluster'] = c

        return dataframe

    def populate_entry_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        dataframe.loc[
            (
                (dataframe.cluster.shift(1) < dataframe.cluster)
            ),
            'enter_long'
        ] = 1

        dataframe.loc[
            (
                (dataframe.cluster.shift(1) > dataframe.cluster)
            ),
            'enter_short'
        ] = 1

        return dataframe

    def populate_exit_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        return dataframe

    def position_size(self, total_asset, risk):

        size = total_asset * self.total_risk / risk if risk > self.total_risk else total_asset

        return size

    def custom_stake_amount(self, pair: str, current_time: datetime, current_rate: float,
                            proposed_stake: float, min_stake: Optional[float], max_stake: float,
                            leverage: float, entry_tag: Optional[str], side: str,
                            **kwargs) -> float:

        dataframe, _ = self.dp.get_analyzed_dataframe(pair=pair, timeframe=self.timeframe)
        prev_close = dataframe.close.iat[-2]
        levels = self.pair_levels(pair)
        
        if side == 'long':
            levels = np.append(levels, dataframe.low.iat[-2])
            levels = np.sort(levels)
            risk = 1 - levels[levels < prev_close][-2] / prev_close
        else:
            levels = np.append(levels, dataframe.high.iat[-2])
            levels = np.sort(levels)
            risk = levels[levels > prev_close][1] / prev_close - 1
        
        stake = self.position_size(max_stake, risk)

        return stake

    def custom_stoploss(self, pair: str, trade: 'Trade', current_time: datetime,
                        current_rate: float, current_profit: float, after_fill: bool, 
                        **kwargs) -> Optional[float]:

        dataframe, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
        prev_close = dataframe.close.iat[-2]
        levels = self.pair_levels(pair)
        stop = trade.get_custom_data(key='stop')

        if stop:
            if trade.is_short:
                levels = np.append(levels, stop)
                levels = np.sort(levels)
                return stoploss_from_absolute(
                    levels[levels > prev_close][1],
                    prev_close,
                    is_short=trade.is_short,
                    leverage=trade.leverage
                )
            levels = np.append(levels, stop)
            levels = np.sort(levels)
            return stoploss_from_absolute(
                    levels[levels < prev_close][-2],
                    prev_close,
                    is_short=trade.is_short,
                    leverage=trade.leverage
                )
        
        return -1

    def order_filled(self, pair: str, trade: Trade, order: 'Order', current_time: datetime, **kwargs) -> None:

        dataframe, _ = self.dp.get_analyzed_dataframe(trade.pair, self.timeframe)
        prev_candle = dataframe.iloc[-2].squeeze()

        if trade.nr_of_successful_entries == 1:
            if trade.is_short:
                trade.set_custom_data(key='stop', value=prev_candle['high'])
            else:
                trade.set_custom_data(key='stop', value=prev_candle['low'])

        return None
    
    def bot_loop_start(self, current_time: datetime, **kwargs) -> None:

        pairs = self.dp.current_whitelist()

        if self.config['runmode'].value in ('live'):
            if self.wallets:
                for pair in pairs:
                    ticker = self.dp.ticker(pair)
                    self.dp.send_msg(self.wallets.get_total(ticker))
                self.dp.send_msg(self.wallets.get_total('USDT'))
        
        for pair in pairs:
            dataframe, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
            if not dataframe.empty:
                prev_close = dataframe.close.iat[-2]
                levels = self.pair_levels(pair)
                if prev_close > levels[-1]:
                    self.dp.send_msg('Price crossed above cluster')
                
                if prev_close < levels[0]:
                    self.dp.send_msg('Price crossed below cluster')

In [ ]:
def pair_levels(self, pair):
    
    # dataframe = load_pair_history(
    #     datadir = self.config["datadir"],
    #     timeframe = self.timeframe,
    #     pair = pair,
    #     data_format = "feather",
    #     candle_type=CandleType.FUTURES,
    # )

    # dataframe['label_1'] = self.cluster(dataframe)

    # grouped = dataframe.groupby(['label_1']).apply(self.cluster).to_dict()
    # for c, values in grouped.items():
    #     condition_1 = dataframe['label_1'] == c
    #     dataframe.loc[condition_1, 'label_2'] = values

    # grouped = dataframe.groupby(['label_1','label_2']).apply(self.cluster).to_dict()
    # for c, values in grouped.items():
    #     condition_1 = dataframe['label_1'] == c[0]
    #     condition_2 = dataframe['label_2'] == c[1]
    #     dataframe.loc[(condition_1 & condition_2), 'label_3'] = values

    # grouped = dataframe.groupby(['label_1','label_2','label_3']).apply(self.cluster).to_dict()
    # for c, values in grouped.items():
    #     condition_1 = dataframe['label_1'] == c[0]
    #     condition_2 = dataframe['label_2'] == c[1]
    #     condition_3 = dataframe['label_3'] == c[2]
    #     dataframe.loc[(condition_1 & condition_2 & condition_3), 'label_4'] = values

    # levels = dataframe.groupby([f'label_{i}' for i in [1,2,3,4]]).min().close.sort_values().values
    # levels = np.append(levels, dataframe.close.max())

    levels = np.genfromtxt('user_data/notebooks/levels.csv', delimiter=',')

    return levels

In [ ]:
from freqtrade.persistence import Trade
# ...
open_trades = Trade.get_open_trade_count()
open_trades

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import datetime

In [2]:
con = sqlite3.connect("../cluster_strategy_V2_dry.sqlite")

In [3]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", con)
tables

,name
0,KeyValueStore
1,trades
2,pairlocks
3,trade_custom_data
4,orders


In [4]:
trade_custom_data = pd.read_sql_query("SELECT * FROM trade_custom_data", con)
print(trade_custom_data.to_markdown())

|    |   id |   ft_trade_id | cd_key   | cd_type   | cd_value                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [44]:
trades = pd.read_sql_query("SELECT * FROM trades", con)

In [45]:
print(trades.tail().to_markdown())

| id   | exchange   | pair   | base_currency   | stake_currency   | is_open   | fee_open   | fee_open_cost   | fee_open_currency   | fee_close   | fee_close_cost   | fee_close_currency   | open_rate   | open_rate_requested   | open_trade_value   | close_rate   | close_rate_requested   | realized_profit   | close_profit   | close_profit_abs   | stake_amount   | max_stake_amount   | amount   | amount_requested   | open_date   | close_date   | stop_loss   | stop_loss_pct   | initial_stop_loss   | initial_stop_loss_pct   | is_stop_loss_trailing   | max_rate   | min_rate   | exit_reason   | exit_order_status   | strategy   | enter_tag   | timeframe   | trading_mode   | amount_precision   | price_precision   | precision_mode   | contract_size   | leverage   | is_short   | liquidation_price   | interest_rate   | funding_fees   | funding_fee_running   |
|------|------------|--------|-----------------|------------------|-----------|------------|-----------------|---------------------|----------

In [12]:
today = datetime.datetime.today().strftime("%Y-%m-%d")
yesterday = (datetime.datetime.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")
yesterday_trades = trades[(trades.close_date <= today) & (trades.close_date >= yesterday)]
print(yesterday_trades.tail().to_markdown())

|    |   id | exchange   | pair          | base_currency   | stake_currency   |   is_open |   fee_open |   fee_open_cost | fee_open_currency   |   fee_close |   fee_close_cost | fee_close_currency   |   open_rate |   open_rate_requested |   open_trade_value |   close_rate |   close_rate_requested |   realized_profit |   close_profit |   close_profit_abs |   stake_amount |   max_stake_amount |   amount |   amount_requested | open_date                  | close_date                 |   stop_loss |   stop_loss_pct |   initial_stop_loss |   initial_stop_loss_pct |   is_stop_loss_trailing |   max_rate |   min_rate | exit_reason        | exit_order_status   | strategy          | enter_tag   |   timeframe | trading_mode   |   amount_precision |   price_precision |   precision_mode |   contract_size |   leverage |   is_short |   liquidation_price |   interest_rate |   funding_fees |   funding_fee_running |
|---:|-----:|:-----------|:--------------|:----------------|:-----------------|----------

In [4]:
class Test:
    def return_name(self):
        print(__class__.__name__)

In [5]:
test = Test()

In [6]:
test.return_name()

Test
